In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from ssf.Taxonomy import Taxonomy
from ssf.Configs import load_config
from ssf.metric_executors import MetricExecutor
from ssf.Constants import *
import pingouin as pg
from statsmodels.stats.multitest import multipletests
import numpy as np

config = load_config(REPLICATION_CONFIG_PATH)
taxonomy = Taxonomy(TAXONOMY_DIR)

# Output directory for ablation results (from config)
ABLATIONS_DIR = config.dirs.data.ablations
os.makedirs(ABLATIONS_DIR, exist_ok=True)

# Force redo flag from config
FORCE_REDO = config.force_redo.ablations

METRICS = ['cosine_similarity']

ablations_csv_path = f'{ABLATIONS_DIR}/ablations.csv'
if os.path.exists(ablations_csv_path) and not FORCE_REDO:
    print(f"Loading precomputed metrics from {ablations_csv_path}...")
    ssf_test_df = pd.read_csv(ablations_csv_path)
else:
    if FORCE_REDO:
        print(f"Force redo enabled, recomputing metrics...")
    else:
        print(f"{ablations_csv_path} not found, computing metrics...")
    ssf_test_df = pd.read_csv(f'{config.dirs.data.corpus}/ssf_split_test.csv')
    metric_executor = MetricExecutor(sbert_model=config.models.sbert_model)
    
    all_results = []
    for dim in tqdm(taxonomy.get_dims(), desc="Processing dimensions"):
        ref_cols = [f"{config.models.openai_default.replace("-", "_")}_{dim}_gen{i}" for i in range(3)]
        cand_cols = [f"{prompt_col_suffix}${dim}_gen0" for prompt_col_suffix in ALL_PROMPT_COL_SUFFIXES]
        
        # Loop through each candidate column
        for cand_col in cand_cols:
            for metric_name in METRICS:
                metric_col_name = f"{cand_col}_{metric_name}"
                metric_values = []
                
                for idx, row in tqdm(ssf_test_df.iterrows()):
                    cand_val = row.get(cand_col)
                    
                    refs = []
                    for ref_col in ref_cols:
                        if ref_col in ssf_test_df.columns:
                            ref_val = row.get(ref_col)
                            if pd.notna(ref_val) and ref_val != "":
                                refs.append(str(ref_val))

                    # Use taxonomy to extract variable values
                    pred_varVals = taxonomy.get_var_vals(dim, str(cand_val))
                    refs_varVals = [taxonomy.get_var_vals(dim, ref) for ref in refs]
                    refs_varVals = [r for r in refs_varVals if r != ["ERROR"]]      # skip parsing failures
                    # Skip if parsing failed
                    if pred_varVals == ["ERROR"] or len(refs_varVals) == 0:
                        metric_values.append(None)
                        print("UNRECOVERABLE PARSING FAILURE")
                        continue
                    
                    # compute metric
                    metric_fn = getattr(metric_executor, metric_name)
                    score = metric_fn(pred_varVals, refs_varVals)
                    metric_values.append(score)
                
                # Add metric column to dataframe
                ssf_test_df[metric_col_name] = metric_values

    print("Metric computation complete!")
    ssf_test_df[cols_to_save].to_csv(ablations_csv_path, index=False)

Loading precomputed metrics from ../data/replication/ablations/ablations.csv...


In [2]:
# Build summary table: avg metric value across dimensions for each prompt type, per metric
for metric_name in METRICS:
    summary_data = []
    
    for prompt_suffix in ALL_PROMPT_COL_SUFFIXES:
        # Collect all metric values across all dimensions for this prompt type
        metric_vals = []
        for dim in taxonomy.get_dims():
            cand_col = f"{prompt_suffix}${dim}_gen0"
            metric_col = f"{cand_col}_{metric_name}"
            if metric_col in ssf_test_df.columns:
                vals = ssf_test_df[metric_col].dropna().tolist()
                metric_vals.extend(vals)
        
        # Compute average across dimensions
        avg_val = sum(metric_vals) / len(metric_vals) if metric_vals else None
        summary_data.append({
            'prompt_type': prompt_suffix,
            f'avg_{metric_name}': avg_val
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_path = f"{ABLATIONS_DIR}/ablations_summary_{metric_name}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved {summary_path}")
    display(summary_df)

Saved ../data/replication/ablations/ablations_summary_cosine_similarity.csv


,prompt_type,avg_cosine_similarity
0,prompt_default,0.541778
1,prompt_noSubredditName_noSubredditDescription_...,0.517169
2,prompt_noSubredditName_noSubredditDescription_...,0.539098
3,prompt_noProgenitorSummary_noConversationSummary,0.521159


In [3]:
# T-tests for cosine similarity scores between prompt conditions
# One-sided t-tests assuming more context leads to better (higher) scores

# For paired t-tests, we need to align scores by row/dimension
def get_paired_scores(prompt_suffix1, prompt_suffix2, metric_name):
    """Get paired scores for two prompt types (only where both have values)."""
    scores1 = []
    scores2 = []
    for dim in taxonomy.get_dims():
        col1 = f"{prompt_suffix1}${dim}_gen0_{metric_name}"
        col2 = f"{prompt_suffix2}${dim}_gen0_{metric_name}"
        if col1 in ssf_test_df.columns and col2 in ssf_test_df.columns:
            for idx, row in ssf_test_df.iterrows():
                val1 = row.get(col1)
                val2 = row.get(col2)
                if pd.notna(val1) and pd.notna(val2):
                    scores1.append(val1)
                    scores2.append(val2)
    return scores1, scores2

# Define comparisons: (more_context_condition, less_context_condition)
# We test if more_context > less_context (alternative='greater')
comparisons = [
    # partial ablations vs full ablation
    ('comm_ctx vs no_ctx', PROMPT_COL_SUFFIX_NO_CONVERSATION_CONTEXT, PROMPT_COL_SUFFIX_NO_CONTEXT),
    ('conv_ctx vs no_ctx', PROMPT_COL_SUFFIX_NO_COMMUNITY_CONTEXT, PROMPT_COL_SUFFIX_NO_CONTEXT),
    # default vs partial ablations
    ('comm+conv_ctx vs comm_ctx', PROMPT_COL_SUFFIX_FULL_CONTEXT, PROMPT_COL_SUFFIX_NO_CONVERSATION_CONTEXT),
    ('comm+conv_ctx vs conv_ctx', PROMPT_COL_SUFFIX_FULL_CONTEXT, PROMPT_COL_SUFFIX_NO_COMMUNITY_CONTEXT),
    # default vs full ablation
    ('comm+conv_ctx vs no_ctx', PROMPT_COL_SUFFIX_FULL_CONTEXT, PROMPT_COL_SUFFIX_NO_CONTEXT),
]

metric_name = 'cosine_similarity'
ttest_results = []

for comparison_name, cond1, cond2 in comparisons:
    s1, s2 = get_paired_scores(cond1, cond2, metric_name)
    # One-sided test: cond1 (more context) > cond2 (less context)
    # pingouin.ttest returns a DataFrame with T, dof, alternative, p-val, CI95%, cohen-d, BF10, power
    result = pg.ttest(s1, s2, paired=True, alternative='greater')
    ttest_results.append({
        'comparison': comparison_name,
        'condition1': cond1,
        'condition2': cond2,
        'n': len(s1),
        'mean1': np.mean(s1),
        'mean2': np.mean(s2),
        't': result['T'].values[0],
        'p': result['p-val'].values[0],
        'cohen_d': result['cohen-d'].values[0]
    })

ttest_df = pd.DataFrame(ttest_results)

# Apply Holm-Bonferroni correction within this metric
p_values = ttest_df['p'].values
rejected, p_holm, _, _ = multipletests(p_values, method='holm')
ttest_df['p_holm'] = p_holm

print("T-test Results (One-sided paired t-tests on cosine similarity scores):")
print("H1: more context condition > less context condition")
display(ttest_df)

# Save to ablation_ttests.csv with proper formatting
ttest_display_df = ttest_df[['comparison', 'n', 'mean1', 'mean2', 't', 'cohen_d', 'p_holm']].copy()
ttest_display_df['mean1'] = ttest_display_df['mean1'].round(3)
ttest_display_df['mean2'] = ttest_display_df['mean2'].round(3)
ttest_display_df['t'] = ttest_display_df['t'].round(3)
ttest_display_df['cohen_d'] = ttest_display_df['cohen_d'].round(3)
ttest_display_df['p_holm'] = ttest_display_df['p_holm'].apply(lambda x: f'{x:.3e}' if x < 0.001 else f'{x:.3f}')
ttest_csv_path = f"{ABLATIONS_DIR}/ablation_ttests.csv"
ttest_display_df.to_csv(ttest_csv_path, index=False)
print(f"\nT-test results saved to {ttest_csv_path}")

# Generate LaTeX table with p_holm < 0.05 in bold
def escape_latex(text):
    """Escape special LaTeX characters"""
    replacements = {
        '_': '\\_',
        '&': '\\&',
        '%': '\\%',
        '$': '\\$',
        '#': '\\#',
        '{': '\\{',
        '}': '\\}',
        '~': '\\textasciitilde{}',
        '^': '\\textasciicircum{}',
    }
    for char, escaped in replacements.items():
        text = text.replace(char, escaped)
    return text

def format_p_latex(p_val):
    """Format p-value for LaTeX, bold if < 0.05"""
    if p_val < 0.001:
        p_str = f'{p_val:.2e}'
    else:
        p_str = f'{p_val:.3f}'
    if p_val < 0.05:
        return f'\\textbf{{{p_str}}}'
    return p_str

latex_rows = []
for _, row in ttest_df.iterrows():
    comparison = escape_latex(row['comparison'])
    n = row['n']
    mean1 = f"{row['mean1']:.3f}"
    mean2 = f"{row['mean2']:.3f}"
    t = f"{row['t']:.3f}"
    cohen_d = f"{row['cohen_d']:.3f}"
    p_holm_formatted = format_p_latex(row['p_holm'])
    latex_rows.append(f"    {comparison} & {n} & {mean1} & {mean2} & {t} & {cohen_d} & {p_holm_formatted} \\\\")

latex_table = """\\begin{table*}[h]
\\centering
\\small
\\begin{tabular}{lcccccc}
\\toprule
comparison & $n$ & $\\bar{x}_1$ & $\\bar{x}_2$ & $t$ & $d$ & $p_{\\text{holm}}$ \\\\
\\midrule
""" + "\n".join(latex_rows) + """
\\bottomrule
\\end{tabular}
\\caption{\\ssfGenerator SFT Distillation Context Ablation Results. We compare \\ssfGenerator variants' inferences against GPT-4o reference inferences via one-sided paired t-tests of cosine similarities, based on the hypothesis that more context will lead to greater inference similarity than less context. Statistically significant results (Holm–Bonferroni corrected) are in \\textbf{bold}.}
\\label{tab:ablation_ttests}
\\end{table*}"""

print("\nLaTeX Table:")
print(latex_table)

# Save LaTeX table to file
latex_path = f"{ABLATIONS_DIR}/ablation_ttests.tex"
with open(latex_path, "w") as f:
    f.write(latex_table)
print(f"\nLaTeX table saved to {latex_path}")

T-test Results (One-sided paired t-tests on cosine similarity scores):
H1: more context condition > less context condition


,comparison,condition1,condition2,n,mean1,mean2,t,p,cohen_d,p_holm
0,comm_ctx vs no_ctx,prompt_noProgenitorSummary_noConversationSummary,prompt_noSubredditName_noSubredditDescription_...,2865,0.521174,0.516912,2.251040,1.222927e-02,0.030392,2.445855e-02
1,conv_ctx vs no_ctx,prompt_noSubredditName_noSubredditDescription_...,prompt_noSubredditName_noSubredditDescription_...,2943,0.539262,0.517157,11.068024,3.164498e-28,0.159135,1.265799e-27
2,comm+conv_ctx vs comm_ctx,prompt_default,prompt_noProgenitorSummary_noConversationSummary,2877,0.542001,0.521143,10.268600,1.276462e-24,0.150982,3.829387e-24
3,comm+conv_ctx vs conv_ctx,prompt_default,prompt_noSubredditName_noSubredditDescription_...,2959,0.541843,0.539202,1.522226,6.402973e-02,0.019295,6.402973e-02
4,comm+conv_ctx vs no_ctx,prompt_default,prompt_noSubredditName_noSubredditDescription_...,2947,0.541958,0.517160,12.078216,4.051842e-33,0.178471,2.025921e-32



T-test results saved to ../data/replication/ablations/ablation_ttests.csv

LaTeX Table:
\begin{table*}[h]
\centering
\small
\begin{tabular}{lcccccc}
\toprule
comparison & $n$ & $\bar{x}_1$ & $\bar{x}_2$ & $t$ & $d$ & $p_{\text{holm}}$ \\
\midrule
    comm\_ctx vs no\_ctx & 2865 & 0.521 & 0.517 & 2.251 & 0.030 & \textbf{0.024} \\
    conv\_ctx vs no\_ctx & 2943 & 0.539 & 0.517 & 11.068 & 0.159 & \textbf{1.27e-27} \\
    comm+conv\_ctx vs comm\_ctx & 2877 & 0.542 & 0.521 & 10.269 & 0.151 & \textbf{3.83e-24} \\
    comm+conv\_ctx vs conv\_ctx & 2959 & 0.542 & 0.539 & 1.522 & 0.019 & 0.064 \\
    comm+conv\_ctx vs no\_ctx & 2947 & 0.542 & 0.517 & 12.078 & 0.178 & \textbf{2.03e-32} \\
\bottomrule
\end{tabular}
\caption{\ssfGenerator SFT Distillation Context Ablation Results. We compare \ssfGenerator variants' inferences against GPT-4o reference inferences via one-sided paired t-tests of cosine similarities, based on the hypothesis that more context will lead to greater inference similarity

In [4]:
# Per-dimension breakdown: avg metric for each prompt type x dimension
per_dim_data = []

for dim in taxonomy.get_dims():
    for prompt_suffix in ALL_PROMPT_COL_SUFFIXES:
        row_data = {'dimension': dim, 'prompt_type': prompt_suffix}
        
        for metric_name in METRICS:
            cand_col = f"{prompt_suffix}${dim}_gen0"
            metric_col = f"{cand_col}_{metric_name}"
            if metric_col in ssf_test_df.columns:
                avg_val = ssf_test_df[metric_col].mean()
            else:
                avg_val = None
            row_data[metric_name] = avg_val
        
        per_dim_data.append(row_data)

per_dim_df = pd.DataFrame(per_dim_data)

# Pivot for better visualization - one table per metric
for metric_name in METRICS:
    print(f"\n=== {metric_name.upper()} ===")
    pivot = per_dim_df.pivot(index='prompt_type', columns='dimension', values=metric_name)
    display(pivot.round(3))


=== COSINE_SIMILARITY ===


dimension,aesthetic_feeling,author_emotional_response,causal_explanation,character_appraisal,moral,narrative_feeling,narrative_intent,overall_goal,prediction,stance
prompt_type,,,,,,,,,,
prompt_default,0.432,0.407,0.497,0.658,0.537,0.536,0.556,0.567,0.616,0.611
prompt_noProgenitorSummary_noConversationSummary,0.424,0.392,0.488,0.635,0.519,0.523,0.513,0.537,0.600,0.582
prompt_noSubredditName_noSubredditDescription_noSubredditValues,0.429,0.400,0.493,0.667,0.544,0.541,0.544,0.572,0.614,0.586
prompt_noSubredditName_noSubredditDescription_noSubredditValues_noProgenitorSummary_noConversationSummary,0.422,0.381,0.482,0.647,0.518,0.518,0.506,0.523,0.592,0.583
